[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Vectorized Math &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


**1.** Convert Tromso's readings, row 3, to Fahrenheit, round them to whole degrees with `np.round`, and print the first five.

In [2]:
tromso_fahrenheit = np.round(readings[3] * 9 / 5 + 32)

print(tromso_fahrenheit[:5])

[20. 17. 17. 17. 18.]


The result is still `float64`, so the whole degrees print with a decimal point.

**2.** Print the saturation pressure at -20, -10 and 0 degrees with `saturation_pressure`, rounded to two places.

In [3]:
def saturation_pressure(celsius):
    return 6.112 * np.exp(17.62 * celsius / (243.12 + celsius))

print(saturation_pressure(np.array([-20.0, -10.0, 0.0])).round(2))

[1.26 2.87 6.11]


The solutions notebook defines the function again, since it is a notebook of its own. Cold air holds much less water vapor, a little over a fifth as much at -20 degrees as at 0.

**3.** Use `np.clip` to keep Oslo's readings, row 1, between 0 and 15, and print the smallest and largest value before and after.

In [4]:
oslo = readings[1]
kept = np.clip(oslo, 0, 15)

print(oslo.min(), oslo.max())
print(kept.min(), kept.max())

-6.3 19.3
0.0 15.0


Oslo has no missing hours, so `.min()` and `.max()` work without their `nan` versions.

**4.** Make an array holding the colder of Bergen and Tromso in every hour with `np.minimum`, and print its mean, rounded to two places.

In [5]:
colder = np.minimum(readings[0], readings[3])

print(colder.mean().round(2))
print((readings[3] < readings[0]).all())

3.5
True


Tromso was colder than Bergen in every hour of the year, so the colder of the two is Tromso's own row, with Tromso's mean.

**5.** Label each of Oslo's days `"warm"` when its mean is above 10 degrees and `"cool"` otherwise with `np.where`, and count the warm days.

In [6]:
oslo_days = readings[1].reshape(365, 24).mean(axis=1)
day_labels = np.where(oslo_days > 10, "warm", "cool")

print((day_labels == "warm").sum())

137


Oslo has no missing days, so every day gets one of the two labels from its own mean.

---

&#8592; **Back to:** [Vectorized Math](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/09-vectorized-math.ipynb)  &nbsp;&middot;&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
